## Overview

CNN-GRU Setup for prediction out of 2D timeseries data

finding out of impact of training on "wrong" year of the saison


Colab version of local, loading data from existing dataloaders

----
## Data:

-2D Space - Timeseries

-predicting 1 feature out of 7 variables

-Forecasting 1 timestep (not the following)

-------
Peter Resch, 6.6.

In [1]:
from __future__ import print_function, division   # Ensures Python3 printing & division standard
import pandas as pd
from pandas import Series, DataFrame
from matplotlib import pyplot as plt
import numpy as np
import os

from pathlib import Path

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
import sklearn
from sklearn.model_selection import train_test_split

import xarray as xr
rSeed=42

SavePlots = False

## Loading Data

In [4]:
from google.colab import drive
drive.mount('/content/drive/')

import sys
sys.path.insert(0, '/content/drive/MyDrive/small_grid/') # Add the directory containing the module to the Python path

import my_dataloader_module

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [5]:
!cd drive/MyDrive/small_grid && ls

dataloaders  models  my_dataloader_module.py  __pycache__


In [9]:
seasons={"spring": "MAM", "summer": "JJA", "autumn": "SON", "winter": "DJF"}

data_dir='drive/MyDrive/small_grid'
!cd {data_dir} && ls

dataloader_path = str(data_dir+ "/dataloaders/")

dataloaderlist = [p.name for p in Path(dataloader_path).iterdir() if p.is_file()]
#clean dataloaderlist, let only .pt files
dataloaderlist = [f for f in dataloaderlist if f.endswith('.pt')]
dataloaderlist=[f for f in dataloaderlist if "dataloader" in f]      #select only the ones with "dataloader" in the name

dataloaders  models  my_dataloader_module.py  __pycache__


## Loading Dataloaders

In [10]:
dataloaders = {}
i=0
for dl in dataloaderlist:
    time,aim = dl.split("_dataloader_")
    time,season=time.split("_")
    aim = aim.split(".pt")[0]
    #print(time, season, aim)
    dataloaders[time,season,aim] = torch.load(dataloader_path + time + "_" + season + "_dataloader_"+ aim+".pt", weights_only=False)
    i=i+1
print(i,"Dataloaders loaded successfully.")

20 Dataloaders loaded successfully.


## Building the Neural Network

In [11]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")


hidden_size=16

class CNN_GRU(nn.Module):
    def __init__(self,in_channels=7,hidden_size=16,lat_size=5,lon_size=5):
        super().__init__()

        #compressing space
        def double_conv(in_ch, out_ch):
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 3, padding=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_ch, out_ch, 3, padding=1),
                nn.ReLU(inplace=True)
            )
        self.conv1 = double_conv(in_channels, 16)
        self.maxpool = nn.MaxPool2d(kernel_size=(2, 2))
        self.conv2 = double_conv(16, 32)

        #compressing time
        self.gru = nn.GRU(input_size=32,hidden_size=32*25, num_layers=1,dropout=0.2,batch_first=True)

        #expanding space
        self.deconv1 = nn.ConvTranspose2d(32, 16, kernel_size=3, padding=1)
        self.deconv2 = nn.ConvTranspose2d(16, 1, kernel_size=3, padding=1)


    def forward(self, x):
        #print("starting forward:",x.shape)
        x_gru=[]
        #recognize patterns of the spatial data with CNN
        for time in range(x.shape[1]):
            #print(time)
            cnn_in=x[:, time, :, :]
            #print("time,cnn_in.shape:",time,cnn_in.shape)
            cnn_in=self.conv1(cnn_in)
            #print("after conv1:",cnn_in.shape)
            cnn_in = self.maxpool(cnn_in)
            #print("after maxpool:",cnn_in.shape)
            cnn_in = self.conv2(cnn_in)
            #print("after conv2:",cnn_in.shape)
            cnn_in = self.maxpool(cnn_in)
            #print("after maxpool:",cnn_in.shape)
            cnn_in = cnn_in.view(cnn_in.size(0), -1)  # Flatten for GRU input
            #print("after flatten:",cnn_in.shape)
            x_gru.append(cnn_in)
        x_gru = torch.stack(x_gru, dim=1)
        #print("shaped for GRU:",x_gru.shape)#(batch, time, convoluted features with space)

        #decoding the temporal patterns with GRU
        x,_ = self.gru(x_gru)#x:(batch, time, hidden_size)
        x = x[:,-1,:].view(x.shape[0],32,5,5)##(batch, last hidden_size,lat_size,lon_size)
        #print("after GRU:",x.shape)
        x=self.deconv1(x)
        #print("after deconv1:",x.shape)
        x=self.deconv2(x)
        #print("after deconv2:",x.shape)
        return x#


cnn_gru_model=CNN_GRU().to(device)
print(cnn_gru_model)


Using cuda device


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:1364: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__("GRU", *args, **kwargs)


CNN_GRU(
  (conv1): Sequential(
    (0): Conv2d(7, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
  )
  (maxpool): MaxPool2d(kernel_size=(2, 2), stride=(2, 2), padding=0, dilation=1, ceil_mode=False)
  (conv2): Sequential(
    (0): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
  )
  (gru): GRU(32, 800, batch_first=True, dropout=0.2)
  (deconv1): ConvTranspose2d(32, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (deconv2): ConvTranspose2d(16, 1, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
)


In [12]:
def train(dataloader, model, loss_fn, optimizer,device):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        #print(X.shape, y.shape)
        X, y = X.to(device), y.to(device)
        #print(X.shape, y.shape)
        pred = model(X)#.squeeze()
        #print(pred)#.shape)
        loss = loss_fn(pred, y)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            #print(f"Train Loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")
    return loss.item()



def test(dataloader, model, loss_fn,device):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss = 0.0
    all_predictions = []
    all_targets = []

    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            predictions = model(X)
            test_loss += loss_fn(predictions, y).item()

            all_predictions.append(predictions.detach().cpu().numpy())
            all_targets.append(y.detach().cpu().numpy())

    test_loss /= num_batches
    y_pred = np.concatenate(all_predictions)#[:,:,0]
    y_true = np.concatenate(all_targets)#[:,:,0]
    #print(f"y_true shape: {y_true.shape}, y_pred shape: {y_pred.shape}")

    # Flatten everything to 2D: (samples, features)
    y_true_flat = y_true.reshape(y_true.shape[0], -1)
    y_pred_flat = y_pred.reshape(y_pred.shape[0], -1)

    #mae = sklearn.metrics.mean_absolute_error(y_true, y_pred)
    #rmse = np.sqrt(sklearn.metrics.mean_squared_error(y_true, y_pred))
    r2 = sklearn.metrics.r2_score(y_true_flat, y_pred_flat)
    print(f"Test Error:\n R2: {r2:>8f}, Avg loss: {test_loss:>8f} \n")
    return test_loss

## Train the model

In [14]:
epochs = 50

for season in seasons.keys():
    print(":"*50)
    print(f"Season: {season}")

    # Re-initialize the model for each season
    cnn_gru_model = CNN_GRU().to(device)
    model_name = f"cnn_gru_combined_{season}"

    loss_fcn = nn.MSELoss()
    optimizer_cnn_gru = AdamW(cnn_gru_model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer_cnn_gru, mode="min", factor=0.5, patience=2
    )
    train_losses = []
    val_losses = []
    test_losses = []

    # Combine 'past' and 'recent' dataloaders for training and validation
    past_train_dl = dataloaders["past", season, "train"]
    recent_train_dl = dataloaders["recent", season, "train"]
    combined_train_dataset = torch.utils.data.ConcatDataset([past_train_dl.dataset, recent_train_dl.dataset])
    # Assuming batch size and shuffle are consistent or should be based on one of the original Dataloaders
    combined_train_dataloader = DataLoader(combined_train_dataset, batch_size=past_train_dl.batch_size, shuffle=True)

    past_val_dl = dataloaders["past", season, "val"]
    recent_val_dl = dataloaders["recent", season, "val"]
    combined_val_dataset = torch.utils.data.ConcatDataset([past_val_dl.dataset, recent_val_dl.dataset])
    combined_val_dataloader = DataLoader(combined_val_dataset, batch_size=past_val_dl.batch_size, shuffle=False)

    # The test dataloader remains from 'now' data
    test_dataloader = dataloaders["now", season, "test"]

    for t in range(epochs):
        print(f"Epoch {t+1}\n-------------------------------")
        train_loss = train(combined_train_dataloader, cnn_gru_model, loss_fcn, optimizer_cnn_gru, device)
        val_loss = test(combined_val_dataloader, cnn_gru_model, loss_fcn, device)
        test_loss = test(test_dataloader, cnn_gru_model, loss_fcn, device)

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        test_losses.append(test_loss)

        scheduler.step(val_loss)

    train_losses = np.array(train_losses)
    val_losses = np.array(val_losses)
    test_losses = np.array(test_losses)

    print("Training done!")

    os.makedirs("models", exist_ok=True)
    save_path = f"{data_dir}/models/{model_name}.pth"

    torch.save({
        "model_state_dict": cnn_gru_model.state_dict(),
        "optimizer_state_dict": optimizer_cnn_gru.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "lag_selection": [0, 24, 48, 60, 66, 69, 71],
        "target_var": 4,
        "epoch": epochs
    }, save_path)

    print(f"Saved model to {save_path}")

    df = pd.DataFrame({
        "Train Loss": train_losses,
        "Validate Loss": val_losses,
        "Test Loss": test_losses
    })
    df.to_csv(f"{data_dir}/models/losses_{model_name}.csv", index=False)
    print(f"Saved losses to losses_{model_name}.csv")

    print(f"Finished traininig {model_name} for {epochs} epochs.")
    print("-"*30)


::::::::::::::::::::::::::::::::::::::::::::::::::
Season: spring
Epoch 1
-------------------------------


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:1364: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__("GRU", *args, **kwargs)


Test Error:
 R2: 0.553593, Avg loss: 0.503230 

Test Error:
 R2: 0.545702, Avg loss: 0.472987 

Epoch 2
-------------------------------
Test Error:
 R2: 0.660954, Avg loss: 0.386021 

Test Error:
 R2: 0.646157, Avg loss: 0.371972 

Epoch 3
-------------------------------
Test Error:
 R2: 0.651049, Avg loss: 0.395547 

Test Error:
 R2: 0.649420, Avg loss: 0.369254 

Epoch 4
-------------------------------
Test Error:
 R2: 0.682474, Avg loss: 0.363828 

Test Error:
 R2: 0.676543, Avg loss: 0.341672 

Epoch 5
-------------------------------
Test Error:
 R2: 0.676290, Avg loss: 0.368535 

Test Error:
 R2: 0.671352, Avg loss: 0.346910 

Epoch 6
-------------------------------
Test Error:
 R2: 0.665195, Avg loss: 0.379126 

Test Error:
 R2: 0.673492, Avg loss: 0.343129 

Epoch 7
-------------------------------
Test Error:
 R2: 0.694794, Avg loss: 0.346735 

Test Error:
 R2: 0.703725, Avg loss: 0.312966 

Epoch 8
-------------------------------
Test Error:
 R2: 0.706689, Avg loss: 0.335430 



/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:1364: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__("GRU", *args, **kwargs)


Test Error:
 R2: 0.481797, Avg loss: 0.605977 

Test Error:
 R2: 0.460244, Avg loss: 0.555588 

Epoch 2
-------------------------------
Test Error:
 R2: 0.501392, Avg loss: 0.585829 

Test Error:
 R2: 0.489791, Avg loss: 0.529157 

Epoch 3
-------------------------------
Test Error:
 R2: 0.303621, Avg loss: 0.781318 

Test Error:
 R2: 0.220481, Avg loss: 0.755857 

Epoch 4
-------------------------------
Test Error:
 R2: 0.514673, Avg loss: 0.570878 

Test Error:
 R2: 0.498681, Avg loss: 0.518603 

Epoch 5
-------------------------------
Test Error:
 R2: 0.583904, Avg loss: 0.495043 

Test Error:
 R2: 0.568506, Avg loss: 0.450390 

Epoch 6
-------------------------------
Test Error:
 R2: 0.596991, Avg loss: 0.480507 

Test Error:
 R2: 0.580146, Avg loss: 0.439022 

Epoch 7
-------------------------------
Test Error:
 R2: 0.597934, Avg loss: 0.476102 

Test Error:
 R2: 0.589636, Avg loss: 0.427944 

Epoch 8
-------------------------------
Test Error:
 R2: 0.601877, Avg loss: 0.467313 



/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:1364: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__("GRU", *args, **kwargs)


Test Error:
 R2: 0.600642, Avg loss: 0.365311 

Test Error:
 R2: 0.601573, Avg loss: 0.388190 

Epoch 2
-------------------------------
Test Error:
 R2: 0.628703, Avg loss: 0.339778 

Test Error:
 R2: 0.630947, Avg loss: 0.359124 

Epoch 3
-------------------------------
Test Error:
 R2: 0.623994, Avg loss: 0.344002 

Test Error:
 R2: 0.620906, Avg loss: 0.368731 

Epoch 4
-------------------------------
Test Error:
 R2: 0.635919, Avg loss: 0.332850 

Test Error:
 R2: 0.639270, Avg loss: 0.350961 

Epoch 5
-------------------------------
Test Error:
 R2: 0.605000, Avg loss: 0.361319 

Test Error:
 R2: 0.611713, Avg loss: 0.378047 

Epoch 6
-------------------------------
Test Error:
 R2: 0.608343, Avg loss: 0.358201 

Test Error:
 R2: 0.625188, Avg loss: 0.365786 

Epoch 7
-------------------------------
Test Error:
 R2: 0.631946, Avg loss: 0.335545 

Test Error:
 R2: 0.663737, Avg loss: 0.327791 

Epoch 8
-------------------------------
Test Error:
 R2: 0.705421, Avg loss: 0.268923 



/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:1364: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__("GRU", *args, **kwargs)


Test Error:
 R2: 0.672023, Avg loss: 0.309539 

Test Error:
 R2: 0.682319, Avg loss: 0.307636 

Epoch 2
-------------------------------
Test Error:
 R2: 0.615010, Avg loss: 0.364394 

Test Error:
 R2: 0.632605, Avg loss: 0.357161 

Epoch 3
-------------------------------
Test Error:
 R2: 0.653032, Avg loss: 0.328218 

Test Error:
 R2: 0.666746, Avg loss: 0.323680 

Epoch 4
-------------------------------
Test Error:
 R2: 0.719622, Avg loss: 0.264331 

Test Error:
 R2: 0.735022, Avg loss: 0.256058 

Epoch 5
-------------------------------
Test Error:
 R2: 0.729962, Avg loss: 0.254922 

Test Error:
 R2: 0.742995, Avg loss: 0.249325 

Epoch 6
-------------------------------
Test Error:
 R2: 0.794978, Avg loss: 0.193352 

Test Error:
 R2: 0.802082, Avg loss: 0.191438 

Epoch 7
-------------------------------
Test Error:
 R2: 0.802689, Avg loss: 0.186256 

Test Error:
 R2: 0.811480, Avg loss: 0.182715 

Epoch 8
-------------------------------
Test Error:
 R2: 0.775264, Avg loss: 0.212233 

